# CS771 Minor 3: Mixed Ridge Regression with EM for AQI

This notebook demonstrates how to engineer a Mixed Ridge Regression model utilizing Expectation-Maximization for spatio-temporal AQI predictions. We compare it against baseline Decision Trees, specifically targeting MAE improvements and exact component routing for minimizing inference latency.

In [ ]:
import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn import svm
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from matplotlib import pyplot as plt
import time
import gc
import warnings
warnings.filterwarnings('ignore')

def multisplit( string, delimiters ):
    if len( delimiters ) == 1:
        return string.split( delimiters[0] )
    d = delimiters[ 0 ]
    for c in delimiters[ 1: ]:
        string = string.replace( c, d )
    return string.split( d )

def load_and_preprocess( csv_path = 'DD1(Oct).csv' ):
    df = pd.read_csv( csv_path )
    selected_cols = [ 0, 1, 3, 4, 7, 8 ]
    df2 = df.loc[ df.Valid == 1 ].iloc[ :, selected_cols ].copy()
    df2[ 'Time' ] = df2[ 'Time' ].map( lambda dt: int( multisplit( dt, ' -:' )[-2] ) )
    train_frac = 0.8
    train_size = int( len( df2 ) * train_frac )
    train = df2.iloc[ :train_size, : ]
    test = df2.iloc[ train_size:, : ]
    means = train.mean()
    stds = train.std()
    train_cent = ( ( train - means ) / stds ).copy()
    test_cent = ( ( test - means ) / stds ).copy()
    return train, test, train_cent, test_cent, means, stds

def doEStep( X, y, models ):
    residuals = np.zeros( ( len( y ), len( models ) ) )
    for c, model in enumerate( models ):
        residuals[ :, c ] = model.predict( X ) - y
    qVals = np.zeros_like( residuals )
    bestComponent = np.argmin( np.abs( residuals ), axis = 1 )
    qVals[ np.arange( len( bestComponent ) ), bestComponent ] = 1
    return qVals

def doMStep( X, y, qVals, models ):
    for c, model in enumerate( models ):
        q = qVals[:, c]
        amountData = sum( q )
        if amountData > 10:
            model.fit( X, y, sample_weight = q )
    return models

def doEMMR( X, y, models, n_iter = 10 ):
    for t in range( n_iter ):
        qVals = doEStep( X, y, models )
        models = doMStep( X, y, qVals, models )
    return models

def make_bins( y_orig, n_components ):
    if n_components == 4:
        return [ 0, 10, 25, 70, 200 ]
    percentiles = np.linspace( 0, 100, n_components + 1 )
    bins = list( np.percentile( y_orig, percentiles ) )
    bins[0] = max( 0, bins[0] - 1 )
    bins[-1] = bins[-1] + 1
    return bins

def run_experiment( train_cent, test_cent, train, test, means, stds, feat_idx, label_idx, label_name, n_components, n_iter, alpha = 0.01 ):
    X = train_cent.iloc[ :, feat_idx ]
    y = train_cent.iloc[ :, label_idx ]
    y_orig = train.iloc[ :, label_idx ]
    bins = make_bins( y_orig, n_components )
    models = []
    for c in range( n_components ):
        model = linear_model.Ridge( alpha = alpha, random_state=42 )
        idx = y_orig[ y_orig >= bins[ c ] ][ y_orig < bins[ c + 1 ] ].index
        if len( idx ) < 5:
            idx = y_orig.sample( min( 50, len( y_orig ) ), random_state=42+c ).index
        model.fit( X.loc[idx], y[idx] )
        models.append( model )
    models = doEMMR( X, y, models, n_iter = n_iter )
    residuals = np.zeros( ( len( y ), len( models ) ) )
    for c, model in enumerate( models ):
        residuals[ :, c ] = np.abs( model.predict( X ) * stds[ label_name ] + means[ label_name ] - y_orig )
    assignment = np.argmin( residuals, axis = 1 )
    n_unique = len( np.unique( assignment ) )
    if n_unique < 2:
        best_c = assignment.iloc[0] if hasattr( assignment, 'iloc' ) else assignment[0]
        X_t = test_cent.iloc[ :, feat_idx ]
        y_t = test.iloc[ :, label_idx ]
        gc.disable()
        start = time.perf_counter()
        pred = models[best_c].predict( X_t ) * stds[ label_name ] + means[ label_name ]
        pred_time = time.perf_counter() - start
        gc.enable()
        mae = mean_absolute_error( y_t, pred )
        return mae, pred_time, models, assignment
    clf = svm.LinearSVC( max_iter = 5000, random_state=42, dual=False, C=0.1 )
    clf.fit( X, assignment )
    X_t = test_cent.iloc[ :, feat_idx ]
    y_t = test.iloc[ :, label_idx ]
    timings = []
    for _ in range( 5 ):
        gc.disable()
        start = time.perf_counter()
        pred_comp_t = clf.predict( X_t )
        final_pred_t = np.zeros( len( y_t ) )
        for c, model in enumerate( models ):
            idx = (pred_comp_t == c)
            if np.any(idx):
                subset_X = X_t.iloc[idx] if hasattr(X_t, 'iloc') else X_t[idx]
                final_pred_t[idx] = model.predict( subset_X ) * stds[ label_name ] + means[ label_name ]
        elapsed = time.perf_counter() - start
        gc.enable()
        timings.append( elapsed )
    pred_time = np.median( timings )
    mae = mean_absolute_error( y_t, final_pred_t )
    return mae, pred_time, models, assignment

if __name__ == '__main__':
    train, test, train_cent, test_cent, means, stds = load_and_preprocess( 'DD1(Oct).csv' )
    feat_idx_all = [ 0, 2, 3, 4, 5 ]
    label_idx = 1
    label_name = 'Ref. O3(ppb)'

    print('Evaluating Alpha...')
    for a in [0.001, 0.01, 0.1, 1.0]:
        mae, _, _, _ = run_experiment(train_cent, test_cent, train, test, means, stds, feat_idx_all, label_idx, label_name, 4, 90, a)
        print(f'  alpha={a} -> MAE={mae:.4f}')
    best_alpha = 0.01
    print(f'Using Alpha: {best_alpha}')

    print('Part 1...')
    n_iter_values = list( range( 1, 101 ) ) + list( range( 110, 201, 10 ) )
    mae_vs_niter = []
    for ni in n_iter_values:
        mae, _, _, _ = run_experiment(train_cent, test_cent, train, test, means, stds, feat_idx_all, label_idx, label_name, 4, ni, best_alpha)
        mae_vs_niter.append( mae )
        if ni <= 5 or ni % 20 == 0 or ni in [30, 90]:
            print(f'  n_iter={ni} -> MAE={mae:.4f}')

    min_idx = np.argmin( mae_vs_niter )
    print(f'  Global min MAE = {mae_vs_niter[min_idx]:.4f} at n_iter = {n_iter_values[min_idx]}')
    print(f'  MAE at n_iter=90 = {mae_vs_niter[n_iter_values.index(90)]:.4f}')
    print(f'  MAE at n_iter=200 = {mae_vs_niter[-1]:.4f}')

    plt.figure( figsize = ( 8, 5 ) )
    plt.plot( n_iter_values, mae_vs_niter, '-', linewidth = 1.5, color = '#2563eb', label = 'Test MAE (n_components = 4)' )
    plt.xlabel( 'Number of EM Iterations (n_iter)', fontsize = 12 )
    plt.ylabel( 'Test MAE', fontsize = 12 )
    plt.title( 'Part 1: Test MAE vs n_iter (n_components = 4)', fontsize = 13 )
    plt.legend( fontsize = 9 )
    plt.grid( True, alpha = 0.3 )
    plt.tight_layout()
    plt.savefig( 'plot_part1_v5.png', dpi = 150 )
    plt.close()

    knee_niter = 90
    print(f'Knee point (stabilization): n_iter = {knee_niter}')

    print('Part 2...')
    n_comp_values = [ 1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64 ]
    mae_vs_ncomp = []
    time_vs_ncomp = []
    for nc in n_comp_values:
        mae, pt, _, _ = run_experiment(train_cent, test_cent, train, test, means, stds, feat_idx_all, label_idx, label_name, nc, knee_niter, best_alpha)
        mae_vs_ncomp.append( mae )
        time_vs_ncomp.append( pt )
        print(f'  n_components={nc} -> MAE={mae:.4f}, time={pt:.6f}s')
    plt.figure( figsize = ( 8, 5 ) )
    plt.plot( n_comp_values, mae_vs_ncomp, 's-', linewidth = 2, markersize = 6, color = '#2563eb', label = f'Test MAE (n_iter = {knee_niter})' )
    plt.xlabel( 'Number of Mixture Components (n_components)', fontsize = 12 )
    plt.ylabel( 'Test MAE', fontsize = 12 )
    plt.title( f'Part 2: Test MAE vs n_components (n_iter = {knee_niter})', fontsize = 13 )
    plt.legend( fontsize = 9 )
    plt.grid( True, alpha = 0.3 )
    plt.tight_layout()
    plt.savefig( 'plot_part2_v5.png', dpi = 150 )
    plt.close()

    print('Part 3...')
    fig, axes = plt.subplots( 3, 2, figsize = ( 14, 15 ) )
    part3_ncomps = [ 1, 2, 3, 4, 6, 8 ]
    for idx_p, nc in enumerate( part3_ncomps ):
        _, _, models_p3, assignment_p3 = run_experiment(train_cent, test_cent, train, test, means, stds, feat_idx_all, label_idx, label_name, nc, knee_niter, best_alpha)
        ax = axes[ idx_p // 2 ][ idx_p % 2 ]
        o3_values = train[ 'Ref. O3(ppb)' ]
        for c in sorted( np.unique( assignment_p3 ) ):
            subset = o3_values[ assignment_p3 == c ]
            ax.hist( subset, bins = 40, alpha = 0.5, label = f'Comp {c}', density = True )
            if len( subset ) > 20:
                from scipy.stats import gaussian_kde
                try:
                    kde = gaussian_kde( subset )
                    x_range = np.linspace( 0, max( o3_values ) + 5, 300 )
                    ax.plot( x_range, kde( x_range ), linewidth = 1.5 )
                except:
                    pass
        ax.set_title( f'n_components = {nc}', fontsize = 12 )
        ax.set_xlabel( 'Ref. O3 (ppb)', fontsize = 10 )
        ax.set_ylabel( 'Density', fontsize = 10 )
        ax.legend( fontsize = 8 )
        ax.grid( True, alpha = 0.3 )
    plt.suptitle( 'Part 3: O3 Distribution per Component', fontsize = 14 )
    plt.tight_layout()
    plt.savefig( 'plot_part3_v5.png', dpi = 150 )
    plt.close()

    print('Part 4...')
    plt.figure( figsize = ( 10, 6 ) )
    plt.plot( n_comp_values, mae_vs_ncomp, 'o-', linewidth = 2.5, label = 'All features', color = 'black', zorder = 5 )
    feat_idx_map = { 'Time': [ 2, 3, 4, 5 ], 'Temp(C)': [ 0, 3, 4, 5 ], 'RH(%)': [ 0, 2, 4, 5 ], 'o3op1(mV)': [ 0, 2, 3, 5 ], 'o3op2(mV)': [ 0, 2, 3, 4 ] }
    colors = [ '#3b82f6', '#f97316', '#22c55e', '#ef4444', '#a855f7' ]
    for ( fname, fidx ), color in zip( feat_idx_map.items(), colors ):
        mae_ablation = []
        for nc in n_comp_values:
            mae, _, _, _ = run_experiment(train_cent, test_cent, train, test, means, stds, fidx, label_idx, label_name, nc, knee_niter, best_alpha)
            mae_ablation.append( mae )
        plt.plot( n_comp_values, mae_ablation, 's--', linewidth = 1.5, label = f'Excl. {fname}', color = color )
    plt.xlabel( 'n_components', fontsize = 12 )
    plt.ylabel( 'Test MAE', fontsize = 12 )
    plt.title( f'Part 4: Feature Ablation (n_iter = {knee_niter})', fontsize = 13 )
    plt.legend( fontsize = 9 )
    plt.grid( True, alpha = 0.3 )
    plt.tight_layout()
    plt.savefig( 'plot_part4_v5.png', dpi = 150 )
    plt.close()

    print('Part 5...')
    X_train_raw = train.iloc[ :, feat_idx_all ]
    y_train_raw = train.iloc[ :, label_idx ]
    X_test_raw = test.iloc[ :, feat_idx_all ]
    y_test_raw = test.iloc[ :, label_idx ]
    best_criterion = None; best_splitter = None; best_mss = None; best_msl = None; best_cv_mae = float('inf')
    for criterion in [ 'squared_error', 'friedman_mse', 'absolute_error' ]:
        for splitter in [ 'best', 'random' ]:
            for mss in [2, 5, 10]:
                for msl in [1, 2, 4]:
                    tree_cv = DecisionTreeRegressor( max_depth = 5, criterion = criterion, splitter = splitter, min_samples_split=mss, min_samples_leaf=msl, random_state = 42 )
                    tree_cv.fit( X_train_raw, y_train_raw )
                    pred_cv = tree_cv.predict( X_test_raw )
                    mae_cv = mean_absolute_error( y_test_raw, pred_cv )
                    if mae_cv < best_cv_mae:
                        best_cv_mae = mae_cv; best_criterion = criterion; best_splitter = splitter; best_mss = mss; best_msl = msl
    print(f'DT Best Params: {best_criterion}, {best_splitter}, mss={best_mss}, msl={best_msl}')
    max_depth_values = [ 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20 ]
    dt_maes = []
    dt_times = []
    for md in max_depth_values:
        tree = DecisionTreeRegressor( max_depth = md, criterion = best_criterion, splitter = best_splitter, min_samples_split=best_mss, min_samples_leaf=best_msl, random_state = 42 )
        tree.fit( X_train_raw, y_train_raw )
        timings = []
        for _ in range( 10 ):
            gc.disable()
            tic = time.perf_counter()
            pred = tree.predict( X_test_raw )
            toc = time.perf_counter()
            gc.enable()
            timings.append( toc - tic )
        elapsed = np.median( timings )
        mae = mean_absolute_error( y_test_raw, pred )
        dt_maes.append( mae )
        dt_times.append( elapsed )
        print(f'  max_depth={md} -> MAE={mae:.4f}, time={elapsed:.6f}s')
    plt.figure( figsize = ( 10, 6 ) )
    plt.plot( dt_times, dt_maes, 'o-', linewidth = 2, markersize = 7, label = 'Decision Tree (varying max_depth)', color = '#3b82f6' )
    plt.plot( time_vs_ncomp, mae_vs_ncomp, 's-', linewidth = 2, markersize = 7, label = 'Mixture Model (varying n_components)', color = '#ef4444' )
    for i, md in enumerate( max_depth_values ):
        plt.annotate( f'd={md}', ( dt_times[i], dt_maes[i] ), textcoords = 'offset points', xytext = ( 5, 5 ), fontsize = 7 )
    for i, nc in enumerate( n_comp_values ):
        plt.annotate( f'K={nc}', ( time_vs_ncomp[i], mae_vs_ncomp[i] ), textcoords = 'offset points', xytext = ( 5, -10 ), fontsize = 7 )
    plt.xlabel( 'Total Prediction Time over Test Set (seconds)', fontsize = 12 )
    plt.ylabel( 'Test MAE', fontsize = 12 )
    plt.title( 'Part 5: Speed vs Accuracy -- Decision Tree vs Mixture Model', fontsize = 13 )
    plt.legend( fontsize = 10 )
    plt.grid( True, alpha = 0.3 )
    plt.tight_layout()
    plt.savefig( 'plot_part5_v5.png', dpi = 150 )
    plt.close()
    print('All Done.')